In [2]:
import pandas as pd
from meteostat import Hourly, Point


class WeatherPrep():
    def __init__(
        self,
        lat: float,
        lon: float,
        start,
        end,
    ):
        self._lat = lat
        self._lon = lon
        self._start = start
        self._end = end

        self._df = None

    @property
    def lat(self):
        return self._lat

    @property
    def lon(self):
        return self._lon

    @property
    def start(self):
        return self._start

    @property
    def end(self):
        return self._end

    @property
    def df(self):
        if self._df is None:
            self._df = self._prepare_df()
        return self._df

    def _prepare_df(self):
        point = Point(self.lat, self.lon)
        df = Hourly(point, self.start, self.end).fetch()

        if df.empty:
            raise Exception("No weather data found")

        df = df.interpolate(limit=3)
        df = df.bfill().ffill()

        df = df.reset_index()
        return df

In [7]:
import time
import requests
import pandas as pd


class GeoPrep():
    def __init__(
        self,
        hotel_name: str,
        city: str = None,
        country: str = "France",
        radii_km=(1, 2, 3),
        user_agent: str = "ml-project",
        overpass_url: str = "https://overpass-api.de/api/interpreter",
        timeout: int = 60,
        sleep_seconds: float = 1.2,
        max_retries: int = 3,
    ):
        self._hotel_name = hotel_name
        self._city = city
        self._country = country
        self._radii_km = radii_km
        self._user_agent = user_agent
        self._overpass_url = overpass_url
        self._timeout = timeout
        self._sleep_seconds = sleep_seconds
        self._max_retries = max_retries

        self._location = None
        self._hotel_df = None
        self._poi_df = None

    @property
    def hotel_name(self):
        return self._hotel_name

    @property
    def city(self):
        if self._city is None:
            _ = self.location
        return self._city

    @property
    def country(self):
        return self._country

    @property
    def radii_km(self):
        return self._radii_km

    @property
    def user_agent(self):
        return self._user_agent

    @property
    def overpass_url(self):
        return self._overpass_url

    @property
    def timeout(self):
        return self._timeout

    @property
    def sleep_seconds(self):
        return self._sleep_seconds

    @property
    def max_retries(self):
        return self._max_retries

    @property
    def location(self):
        if self._location is None:
            self._location = self._geocode()
        return self._location

    @property
    def hotel_df(self):
        if self._hotel_df is None:
            loc = self.location
            self._hotel_df = pd.DataFrame([{
                "HOTEL_NAME": self.hotel_name,
                "CITY": self.city,
                "COUNTRY": self.country,
                "ADDRESS": loc["address"],
                "LAT": loc["lat"],
                "LON": loc["lon"],
            }])
        return self._hotel_df

    @property
    def poi_df(self):
        if self._poi_df is None:
            self._poi_df = self._prepare_poi_df()
        return self._poi_df

    def _headers(self):
        return {
            "User-Agent": self.user_agent,
            "Accept": "application/json",
        }

    def _safe_get_json(self, url, params):
        last_error = None

        for attempt in range(1, self.max_retries + 1):
            try:
                r = requests.get(
                    url,
                    params=params,
                    headers=self._headers(),
                    timeout=self.timeout,
                )

                if r.status_code != 200:
                    last_error = Exception(
                        f"HTTP {r.status_code} - {r.text[:300]}"
                    )
                    time.sleep(self.sleep_seconds * attempt)
                    continue

                content_type = r.headers.get("Content-Type", "")
                if "json" not in content_type.lower():
                    last_error = Exception(
                        f"Réponse non JSON. Content-Type={content_type}, body={r.text[:300]}"
                    )
                    time.sleep(self.sleep_seconds * attempt)
                    continue

                return r.json()

            except Exception as e:
                last_error = e
                time.sleep(self.sleep_seconds * attempt)

        raise last_error

    def _extract_city_from_address(self, address_dict):
        return (
            address_dict.get("city")
            or address_dict.get("town")
            or address_dict.get("village")
            or address_dict.get("municipality")
            or address_dict.get("suburb")
            or address_dict.get("county")
        )

    def _clean_name(self, name):
        import unicodedata

        # enlever accents
        name = ''.join(
            c for c in unicodedata.normalize('NFD', name)
            if unicodedata.category(c) != 'Mn'
        )

        # simplification
        name = name.replace("Sacré-Cœur", "Sacre Coeur")
        name = name.replace("-", " ")

        return name


    def _geocode(self):
        url = "https://nominatim.openstreetmap.org/search"

        # différentes tentatives
        queries = []

        base_name = self.hotel_name
        clean_name = self._clean_name(base_name)

        # stratégie fallback
        queries.append(f"{base_name}, {self.country}")
        queries.append(f"{clean_name}, {self.country}")

        # fallback : enlever mots marketing
        words = clean_name.split()
        if len(words) > 3:
            queries.append(f"{' '.join(words[:3])}, {self.country}")

        # fallback : juste les 2 derniers mots (souvent ville incluse)
        if len(words) >= 2:
            queries.append(f"{' '.join(words[-2:])}, {self.country}")

        for query in queries:
            params = {
                "q": query,
                "format": "jsonv2",
                "limit": 1,
                "addressdetails": 1,
            }

            try:
                data = self._safe_get_json(url, params)
                time.sleep(self.sleep_seconds)

                if data:
                    best = data[0]
                    address_dict = best.get("address", {})

                    if self._city is None:
                        self._city = self._extract_city_from_address(address_dict)

                    return {
                        "address": best.get("display_name"),
                        "lat": float(best["lat"]),
                        "lon": float(best["lon"]),
                    }

            except Exception as e:
                continue

        # si tout échoue → fallback sur ville seule
        if self._city:
            query = f"{self._city}, {self.country}"
            params = {
                "q": query,
                "format": "jsonv2",
                "limit": 1,
            }

            data = self._safe_get_json(url, params)

            if data:
                best = data[0]
                return {
                    "address": best.get("display_name"),
                    "lat": float(best["lat"]),
                    "lon": float(best["lon"]),
                }

        raise Exception(f"Hotel not found after fallback: {self.hotel_name}")

    def _build_overpass_query(self, lat, lon, radius_km):
        radius_m = int(radius_km * 1000)

        return f"""
        [out:json][timeout:25];
        (
          node(around:{radius_m},{lat},{lon})["shop"="supermarket"];
          way(around:{radius_m},{lat},{lon})["shop"="supermarket"];
          relation(around:{radius_m},{lat},{lon})["shop"="supermarket"];

          node(around:{radius_m},{lat},{lon})["shop"="convenience"];
          way(around:{radius_m},{lat},{lon})["shop"="convenience"];
          relation(around:{radius_m},{lat},{lon})["shop"="convenience"];

          node(around:{radius_m},{lat},{lon})["shop"="tobacco"];
          way(around:{radius_m},{lat},{lon})["shop"="tobacco"];
          relation(around:{radius_m},{lat},{lon})["shop"="tobacco"];

          node(around:{radius_m},{lat},{lon})["shop"="grocery"];
          way(around:{radius_m},{lat},{lon})["shop"="grocery"];
          relation(around:{radius_m},{lat},{lon})["shop"="grocery"];
        );
        out center tags;
        """

    def _get_poi_for_radius(self, lat, lon, radius_km):
        query = self._build_overpass_query(lat, lon, radius_km)

        try:
            data = self._safe_get_json(
                self.overpass_url,
                {"data": query}
            )
            time.sleep(self.sleep_seconds)

        except Exception as e:
            print(f"[WARN] Overpass failed for radius={radius_km} km: {e}")
            return []

        rows = []
        for el in data.get("elements", []):
            tags = el.get("tags", {})
            poi_lat = el.get("lat", el.get("center", {}).get("lat"))
            poi_lon = el.get("lon", el.get("center", {}).get("lon"))

            rows.append({
                "HOTEL_NAME": self.hotel_name,
                "CITY": self.city,
                "HOTEL_LAT": lat,
                "HOTEL_LON": lon,
                "RAYON_KM": radius_km,
                "OSM_TYPE": el.get("type"),
                "OSM_ID": el.get("id"),
                "NAME": tags.get("name"),
                "SHOP_TYPE": tags.get("shop"),
                "BRAND": tags.get("brand"),
                "ADDRESS_STREET": tags.get("addr:street"),
                "ADDRESS_POSTCODE": tags.get("addr:postcode"),
                "ADDRESS_CITY": tags.get("addr:city"),
                "LAT": poi_lat,
                "LON": poi_lon,
            })

        return rows

    def get_distance(self, lat1, lon1, lat2, lon2, mode="driving"):
        url = f"https://router.project-osrm.org/route/v1/{mode}/{lon1},{lat1};{lon2},{lat2}"
        params = {"overview": "false"}

        r = requests.get(url, params=params, timeout=self.timeout)
        r.raise_for_status()
        data = r.json()

        if not data.get("routes"):
            raise Exception(f"No route found for mode={mode}")

        route = data["routes"][0]

        return {
            "distance_km": route["distance"] / 1000,
            "duration_min": route["duration"] / 60
        }

    def _add_distance(self, hotel_lat, hotel_lon, poi_lat, poi_lon):
        car = self.get_distance(hotel_lat, hotel_lon, poi_lat, poi_lon, "driving")
        foot = self.get_distance(hotel_lat, hotel_lon, poi_lat, poi_lon, "foot")
        cycle = self.get_distance(hotel_lat, hotel_lon, poi_lat, poi_lon, "cycling")

        return {
            "DIST_KM_CAR": car["distance_km"],
            "TIME_MIN_CAR": car["duration_min"],
            "DIST_KM_FOOT": foot["distance_km"],
            "TIME_MIN_FOOT": foot["duration_min"],
            "DIST_KM_CYC": cycle["distance_km"],
            "TIME_MIN_CYC": cycle["duration_min"],
        }

    def _prepare_poi_df(self):
        loc = self.location
        lat = loc["lat"]
        lon = loc["lon"]

        all_rows = []
        for radius in self.radii_km:
            rows = self._get_poi_for_radius(lat, lon, radius)
            all_rows.extend(rows)

        if not all_rows:
            return pd.DataFrame(columns=[
                "HOTEL_NAME", "CITY", "HOTEL_LAT", "HOTEL_LON", "RAYON_KM",
                "OSM_TYPE", "OSM_ID", "NAME", "SHOP_TYPE", "BRAND",
                "ADDRESS_STREET", "ADDRESS_POSTCODE", "ADDRESS_CITY",
                "LAT", "LON"
            ])

        df = pd.DataFrame(all_rows)
        df = df.drop_duplicates(subset=["RAYON_KM", "OSM_TYPE", "OSM_ID"]).reset_index(drop=True)
        return df

In [8]:
import pandas as pd
import numpy as np

class DataPrep():
    def __init__(self, 
        filepath : str,
        dt_col : str = "DATE",
        tm_col : str = "HEURE",
        dttm_col : str = "DATETIME",
    ):
        self._filepath = filepath
        self._dt_col = dt_col
        self._tm_col = tm_col
        self._dttm_col = dttm_col

        self._src_df = None
        self._df = None


    @property
    def filepath(self):
        return self._filepath
    

    @property
    def dt_col(self):
        return self._dt_col
    

    @property
    def tm_col(self):
        return self._tm_col
    

    @property
    def dttm_col(self):
        return self._dttm_col


    @property
    def src_df(self):
        if(self._src_df is None):
            self._src_df = pd.read_csv(self.filepath)
        return self._src_df
    


    @property
    def df(self):
        if(self._df is None):
            df = self.prepare_df(self.src_df, 
                dt_col = self.dt_col, 
                tm_col = self.tm_col, 
                dttm_col = self.dttm_col
            )
            self._df = df
        return self._df
    

    @classmethod
    def prepare_df(cls, 
        df : pd.DataFrame,
        dt_col = "DATE",
        tm_col = "HEURE",
        dttm_col = "DATETIME",
        ref_dt = "2020-01-01",
        statut_col = "STATUT",
    ):
        df[dttm_col] = pd.to_datetime(df[dt_col] + " " + df[tm_col], format = "mixed")

   
        df["MOIS_D_ANNEE"] = df[dttm_col].dt.month
        df["JOUR_DU_MOIS"] = df[dttm_col].dt.day
        df["JOUR_DE_SEMAINE"] = df[dttm_col].dt.dayofweek
        df["JOUR_D_ANNEE"] = df[dttm_col].dt.dayofyear
        df["SEMAINE_D_ANNEE"] = df[dttm_col].dt.isocalendar().week.astype(int)
        df["IS_WEEKEND"] = df["JOUR_DE_SEMAINE"].isin([5, 6]).astype(int)
        df["HEURE_DU_JOUR"] = df[dttm_col].dt.hour
        df[f"JOURS_DEPUIS_{ref_dt.replace('-', '_')}"] = (df[dttm_col] - pd.Timestamp(ref_dt)).dt.days

        df = df[df[statut_col] == "DONE"]
        df = df.drop([
            dt_col, tm_col, statut_col, 
            "CODE EAN", "PRIX HT", "METEO DU JOUR (MOYENNE)", "METEO DU MOIS (MOYENNE)"
            ], 
            axis = 1
        )

        return df

In [9]:
filename = "001.queryVentes.csv"
dp = DataPrep(filename)
dp.src_df.head(3)
dp.df.head(3)

,NOM BOUTIQUE,OPERATEUR,MACHINE,NOM DU PRODUIT,QUANTITE,VAT,PRIX TTC,TYPE,GAMME,MARQUE,...,TEMPERATURE,DATETIME,MOIS_D_ANNEE,JOUR_DU_MOIS,JOUR_DE_SEMAINE,JOUR_D_ANNEE,SEMAINE_D_ANNEE,IS_WEEKEND,HEURE_DU_JOUR,JOURS_DEPUIS_2020_01_01
0,Ibis budget Nice,ADIPOS,SCANNER NICE,TONGS FEMME 100 NOIR,1,20.0,6.0,NON-F&B,ACCESSOIRES,DECATHLON,...,23.1,2023-08-10 12:04:22.555,8,10,3,222,32,0,12,1317
1,Ibis budget Nice,ADIPOS,SCANNER NICE,CASQUETTE ENFANT -MH100,1,20.0,12.0,NON-F&B,ACCESSOIRES,DECATHLON,...,23.1,2023-08-10 18:32:38.901,8,10,3,222,32,0,18,1317
2,Ibis budget Nice,ADIPOS,SCANNER NICE,MASQUE EASYBREATH DE SURFACE ADULTE - 500 BLEU,1,20.0,29.0,NON-F&B,ACCESSOIRES,DECATHLON,...,24.6,2023-08-11 12:11:15.590,8,11,4,223,32,0,12,1318


In [ ]:
poi_df = pd.concat([GeoPrep(hotel_name = hotel_name).poi_df for hotel_name in dp.df["NOM BOUTIQUE"].unique()])
poi_df.to_csv("poi.csv") # 4m12s

[WARN] Overpass failed for radius=2 km: HTTP 504 - <?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Strict//EN"
    "http://www.w3.org/TR/xhtml1/DTD/xhtml1-strict.dtd">
<html xmlns="http://www.w3.org/1999/xhtml" xml:lang="en" lang="en">
<head>
  <meta http-equiv="content-type" content="text/html; charset=utf-8" lan
[WARN] Overpass failed for radius=2 km: HTTP 504 - <?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Strict//EN"
    "http://www.w3.org/TR/xhtml1/DTD/xhtml1-strict.dtd">
<html xmlns="http://www.w3.org/1999/xhtml" xml:lang="en" lang="en">
<head>
  <meta http-equiv="content-type" content="text/html; charset=utf-8" lan
[WARN] Overpass failed for radius=3 km: HTTP 504 - <?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Strict//EN"
    "http://www.w3.org/TR/xhtml1/DTD/xhtml1-strict.dtd">
<html xmlns="http://www.w3.org/1999/xhtml" xml:lang="en" lang="en">
<head>
  <meta http-equi

In [ ]:
poi_

NameError: name 'poi_d' is not defined

In [ ]:
poi_df.drop_duplicates(subset = [""])

(818, 15)

In [ ]:
wp = WeatherPrep(
    lat = ,
    lon = ,
    start = dp.df[dp.dt_col].min(),
    end = dp.df[dp.dt_col].max() + datetime.timedelta(days=1),
)

In [ ]:
wp.df.to_csv("weather.csv")